# Phase 9 — Explainability Layer (FINAL)

Two surfaces, with everything validated in Phase 5/9 baked in:

### 1. Disease Grad-CAM — crop-first, eigen-smoothed, heatmap-only-for-disease
- **Model:** the **C-PD** disease classifier (`iks-disease-plantdoc-crop`) — retrained on leaf **crops**, so it attends to the **leaf**, not the background. (The original full-fine-tune model attended to background; see `disease_diagnosis_summary.ipynb`.)
- **Pipeline:** pretrained **YOLO leaf detector** crops the leaf → C-PD classifies → **eigen-smoothed Grad-CAM** (removes the corner-noise artifact).
- **Rule:** a heatmap is shown **only when a disease is predicted**. Healthy leaves get a clean "✓ Healthy — no disease region" label (a heatmap on a healthy leaf is meaningless/confusing). The model distinguishes healthy vs diseased very reliably (measured in Cell 5b).
- **Honest accuracy framing:** 27-class top-1 ≈ 66.6%, but most errors are within-crop disease **subtype** confusion (e.g. corn rust vs corn blight), not healthy/diseased — so the practically meaningful **healthy-vs-diseased** accuracy is much higher.

### 2. Soil Grad-CAM (3 heads) + retrieved-chunk highlighting
- Soil multi-task model: one heatmap per head (`soil_type` / `moisture` / `texture`).
- Retrieval: lexical query↔chunk term overlap (`**…**` markers) showing *why* each chunk was retrieved.

### Scope (deferred to Phase 11)
- No quantitative pointing-game / faithfulness benchmark, no answer-grounded chunk highlighting — Phase 9 ships the explainability *surfaces*; rigorous evaluation is Phase 11.

### Hard rules
- Local commits only — never `git push`. Models / corpus are read-only (imported, not reimplemented).


In [ ]:
# Cell 2 — clone repo + install dependencies (defensive)
import os
import subprocess
import sys

REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
REPO_PATH = "/content/iks-rag-thesis"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    # don't hard-fail if the local clone diverged — just keep going.
    subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], check=False)

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
print(f"Working directory: {os.getcwd()}")

# Phase 9 adds grad-cam + matplotlib on top of the Phase 7/8 dep set.
DEPS = [
    "chromadb>=0.5,<0.6",
    "sentence-transformers>=3.0,<4.0",
    "transformers>=4.44,<4.50",
    "accelerate>=0.33",
    "bitsandbytes>=0.43",
    "rank-bm25>=0.2.2",
    "datasets>=2.20",
    "huggingface_hub>=0.24",
    "timm>=1.0",
    "pillow",
    "grad-cam>=1.5",
    "matplotlib>=3.7",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], check=True)

# Install ultralytics (YOLO leaf detector) in a SEPARATE pip call. Bundling
# it with the pinned deps above can trip pip's resolver; on its own it's clean.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.0"], check=True)
print("Dependencies installed.")


In [ ]:
# Cell 3 — HF Hub login (private chunks + gated Llama + Phase-5/6 weights)
from huggingface_hub import HfApi, login

login()  # interactive — paste the ankit-iiitdmj write-scope token.

info = HfApi().whoami()
print(f"Logged in as: {info.get('name')}")
assert info.get("name") == "ankit-iiitdmj", (
    "HF token belongs to a different user — Phase 9 needs the same private "
    "datasets and gated model access used in Phase 7/8."
)


In [ ]:
# Cell 4 — GPU + CUDA sanity check
import torch

print(f"torch: {torch.__version__}  cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
else:
    print("WARNING: no GPU. Grad-CAM itself is light, but the Phase 7 Llama is")
    print("not loadable on CPU; the retrieval explanation cells need the LLM,")
    print("so switch the runtime to T4 before continuing.")


In [ ]:
# Cell 5 — Load C-PD disease model + soil engine + YOLO leaf cropper + explainability helpers.
#
# FINAL recipe (everything we validated in Phase 5/9):
#   - disease model = C-PD (`iks-disease-plantdoc-crop`): trained on leaf CROPS, attends to the LEAF.
#   - LeafCropper (pretrained YOLO, conf=0.10): detect leaf -> crop before classifying.
#   - Grad-CAM uses eigen_smooth=True (removes the corner-noise artifact).
#   - "heatmap ONLY for disease": healthy leaves get NO heatmap (a heatmap on a healthy
#     leaf is meaningless/confusing). The model distinguishes healthy vs diseased ~perfectly.
import numpy as np
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from src.disease.infer import DiseaseInferenceEngine
from src.disease.leaf_detect import LeafCropper
from src.explain.gradcam import GradCAMResult, _preprocess_for_gradcam
from src.soil.infer import SoilInferenceEngine

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

disease_engine = DiseaseInferenceEngine(
    model_source="ankit-iiitdmj/iks-disease-plantdoc-crop", device=DEVICE,
)
print(f"Disease engine (C-PD): {disease_engine.num_classes} classes on {disease_engine.device}")
assert not any(n.startswith("class_") and n[6:].isdigit() for n in disease_engine.class_names), (
    "Disease engine class names look like 'class_<i>' placeholders."
)

cropper = LeafCropper(conf=0.10)   # pretrained YOLO leaf detector (loads lazily on first crop)

soil_engine = SoilInferenceEngine(
    model_source="ankit-iiitdmj/iks-soil-multitask-v2", device=DEVICE,
)
print(
    f"Soil engine: heads=[soil_type={len(soil_engine.soil_type_classes)}, "
    f"moisture={len(soil_engine.moisture_classes)}, "
    f"texture={len(soil_engine.texture_classes)}]"
)

# The 10 HEALTHY classes (no disease). The other 17 are diseases.
HEALTHY = {"Apple leaf", "Bell_pepper leaf", "Blueberry leaf", "Cherry leaf", "Peach leaf",
           "Raspberry leaf", "Soyabean leaf", "Strawberry leaf", "Tomato leaf", "grape leaf"}
def is_healthy(name: str) -> bool:
    return name in HEALTHY


def disease_explain(pil_crop):
    """Return (GradCAMResult, healthy_bool).

    Heatmap (eigen-smoothed Grad-CAM) is produced ONLY for a disease
    prediction. For a healthy leaf the overlay is the plain crop (no
    heatmap) — so every heatmap unambiguously means "the disease is here".
    """
    pred = disease_engine.predict(pil_crop).prediction
    t, rgb_u8, rgb_f = _preprocess_for_gradcam(pil_crop, image_size=disease_engine.image_size)
    if is_healthy(pred.class_name):
        return (GradCAMResult(rgb_u8, np.zeros(rgb_u8.shape[:2], np.float32),
                              pred.class_name, pred.confidence, pred.class_index), True)
    mod = disease_engine.model._module if hasattr(disease_engine.model, "_module") else disease_engine.model
    bb = disease_engine.model.get_feature_extractor(); mod.eval()
    tt = t.to(next(mod.parameters()).device).requires_grad_(True)
    g = GradCAM(mod, [bb.blocks[-2]])(
        input_tensor=tt, targets=[ClassifierOutputTarget(int(pred.class_index))],
        eigen_smooth=True,
    )[0]
    return (GradCAMResult(show_cam_on_image(rgb_f, g, use_rgb=True), g,
                          pred.class_name, pred.confidence, pred.class_index), False)

print("LeafCropper + is_healthy() + disease_explain() ready.")


In [ ]:
# Cell 5b — Healthy-vs-diseased accuracy (the practically important metric).
# 27-class top-1 is ~66.6%, but most errors are within-crop disease SUBTYPE
# confusion (e.g. corn rust vs corn blight) — not healthy/diseased. The
# clinically meaningful question "is this leaf diseased?" is answered far
# more accurately, which this cell measures on the full PlantDoc test set.
from datasets import load_dataset

pd_test = load_dataset("ankit-iiitdmj/iks-plantdoc", split="test")

correct = total = diseased_missed = 0
for i in range(len(pd_test)):
    crop, _ = cropper.crop(pd_test[i]["image"].convert("RGB"))
    pred_name = disease_engine.predict(crop).prediction.class_name
    th, ph = is_healthy(pd_test[i]["label"]), is_healthy(pred_name)
    correct += (th == ph); total += 1
    if (not th) and ph:
        diseased_missed += 1   # the dangerous error: a diseased leaf called healthy

print(f"Healthy-vs-diseased accuracy: {correct/total:.1%}  ({correct}/{total})")
print(f"  diseased leaves missed as healthy: {diseased_missed}  (the error that matters most)")
print("\nReport both honestly: 27-class top-1 ~66.6% (fine-grained subtype),")
print("plus this healthy-vs-diseased number (the decision that actually matters).")


In [ ]:
# Cell 5c — Disease leaf-attention SHOWCASE (the hero figure for the supervisor).
# Rule: heatmap ONLY for disease. Diseased leaf -> eigen Grad-CAM on the lesion.
# Healthy leaf -> "Healthy, no disease region" (no heatmap).
import random

import matplotlib.pyplot as plt

# 248/106/22 = confirmed diseased leaves with clean lesion attention; plus 2 healthy.
random.seed(0)
healthy_pick = random.sample([i for i in range(len(pd_test)) if pd_test[i]["label"] in HEALTHY], 2)
SHOWCASE = [248, 106, 22] + healthy_pick

fig, ax = plt.subplots(len(SHOWCASE), 2, figsize=(9, 4.3 * len(SHOWCASE)))
for r, i in enumerate(SHOWCASE):
    crop, _ = cropper.crop(pd_test[i]["image"].convert("RGB"))
    cam, healthy = disease_explain(crop)
    ax[r, 0].imshow(crop)
    ax[r, 0].set_title(f"input  (true: {pd_test[i]['label']})", fontsize=10)
    ax[r, 0].axis("off")
    ax[r, 1].imshow(cam.overlay_rgb)
    if healthy:
        ax[r, 1].set_title(f"✓ HEALTHY — {cam.pred_label} ({cam.pred_conf:.0%})\nno disease region",
                           color="green", fontsize=10)
    else:
        ax[r, 1].set_title(f"⚠ DISEASE — {cam.pred_label} ({cam.pred_conf:.0%})\nGrad-CAM = the lesion",
                           color="darkred", fontsize=10)
    ax[r, 1].axis("off")
plt.tight_layout(); plt.show()
print("Diseased leaves -> heatmap on the lesion. Healthy leaves -> no heatmap (clean 'healthy' label).")


In [ ]:
# Cell 6 — Phase 7 RAG pipeline. Same wiring as Phase 8 Cell 6:
# corpus pulled from the private HF dataset (206 chunks across 4
# books, Gemini re-OCR'd in Phase 3b.2), re-embedded into ChromaDB,
# wrapped in the HybridRetriever (dense + sparse + reranker), and
# composed with a Llama-3.1-8B 4-bit grounded generator.
from collections import Counter

import torch

from src.rag.corpus_loader import build_chroma, load_chunks_from_hf
from src.rag.generator import GroundedGenerator
from src.rag.pipeline import RAGPipeline
from src.rag.retriever import HybridRetriever

EXPECTED_CHUNK_COUNT = 206
EXPECTED_PER_BOOK = {
    "vrikshayurveda": 42,
    "brihat_samhita": 136,
    "krishi_parashara": 13,
    "upavanavinoda": 15,
}

chunks = load_chunks_from_hf()
per_book = Counter(c["book_id"] for c in chunks)
print(f"Loaded {len(chunks)} chunks; per-book breakdown:")
for book, n in sorted(per_book.items(), key=lambda kv: -kv[1]):
    print(f"  {book:<22} {n:>4}")
assert len(chunks) == EXPECTED_CHUNK_COUNT, (
    f"Corpus drift: expected {EXPECTED_CHUNK_COUNT} chunks, got {len(chunks)}."
)
for book, expected in EXPECTED_PER_BOOK.items():
    assert per_book.get(book) == expected, (
        f"Per-book drift: {book} expected {expected}, got {per_book.get(book)}"
    )

collection = build_chroma(chunks, persist_dir="corpus/vector_db")
print(f"ChromaDB ready: collection count = {collection.count()}")
retriever = HybridRetriever(collection, use_dense=True, use_sparse=True, use_reranker=True)
generator = GroundedGenerator(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    load_in_4bit=True,
    temperature=0.2,
    max_new_tokens=512,
    seed=42,
)
generator._ensure_loaded()
torch.cuda.empty_cache()
print(f"Llama loaded. CUDA memory in use: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

rag_pipeline = RAGPipeline(retriever=retriever, generator=generator, default_k=5)
print("RAGPipeline ready.")


In [ ]:
# Cell 7 — Demo inputs: REAL distinct PlantDoc + Phantomfs images.
# The Phase 8 Cell-7 bug (Pillow stand-ins → every sample predicting
# the same prior class) made Grad-CAM heatmaps explain a WRONG label.
# Phase 9 reuses the same three real images and refuses to proceed if
# any path is missing — a Grad-CAM over a wrong-disease placeholder
# is a misleading figure.
#
# Two source paths:
#   - LOCAL  : repo's data/plant_disease/ and data/soil/ trees (the
#              laptop has them; they're gitignored so Colab does NOT).
#   - HF Hub : private datasets ankit-iiitdmj/iks-plantdoc and
#              ankit-iiitdmj/iks-soil-phantomfs. Same images, served
#              from Parquet. The cell auto-detects which to use.
from pathlib import Path

from PIL import Image

from src.integration import CausalPathway

PLANTDOC_LOCAL_ROOT = Path(REPO_PATH) / "data" / "plant_disease" / "plantdoc" / "raw"
PHANTOMFS_LOCAL_ROOT = Path(REPO_PATH) / "data" / "soil" / "phantomfs" / "raw" / "Orignal-Dataset"
DEMO_SCRATCH = Path(REPO_PATH) / "_phase9_demo"
DEMO_SCRATCH.mkdir(exist_ok=True)

# Target (label, local-fallback-file, dataset-source) tuples per sample.
# Picked by scripts/find_phase9_demo_images.py: top-conf samples where
# the model is correct AND the Grad-CAM peak lands in the central 60%
# of the image. Only 3 of 256 test images passed all three filters
# (model attention is mostly cornery on PlantDoc -- a known shortcut
# bias). Healthy classes (Peach leaf, grape leaf) dominate the
# qualified set; Corn rust leaf is the one diseased qualifier.
PLANTDOC_TARGETS = {
    "peach_leaf": (
        "Peach leaf",
        "Peach leaf/peach-leaf-isolated-white-background-39426456.jpg",
    ),
    "grape_leaf": (
        "grape leaf",
        "grape leaf/young-fresh-grape-leaf-picture-id183867740_k=6&m=183867740&s=612x612&w=0&h=Q8_s3Yw7p_VKQjda-mZPl9rN4lDmpVjjbMUk95HtVFk=.jpg",
    ),
    "corn_rust_leaf": (
        "Corn rust leaf",
        "Corn rust leaf/Southern%20corn%20rust.ashx_w=600.jpg",
    ),
}
PHANTOMFS_TARGETS = {
    # HF dataset's class_name column drops the "_Soil" suffix; the
    # local raw folder keeps it. Both are listed per target.
    "alluvial_soil": ("Alluvial", "Alluvial_Soil/1.jpg"),
    "black_soil":   ("Black",   "Black_Soil/1.jpg"),
    "red_soil":     ("Red",     "Red_Soil/1.jpg"),
}


def _resolve_local(root: Path, rel: str) -> Path | None:
    """Return the local copy if present and non-empty, else None."""
    p = root / rel
    if p.is_file() and p.stat().st_size > 0:
        return p
    return None


def _fetch_from_hf(
    dataset_id: str, split: str, label_col: str, label_value: str,
    out_path: Path,
) -> Path:
    """Download the first sample with matching label and save as JPG.

    Cached after first run via the HF datasets library — subsequent
    re-runs reuse the cached parquet, no re-download."""
    from datasets import load_dataset

    if out_path.is_file() and out_path.stat().st_size > 0:
        return out_path
    ds = load_dataset(dataset_id, split=split)
    for sample in ds:
        if sample.get(label_col) == label_value:
            sample["image"].convert("RGB").save(out_path, format="JPEG")
            return out_path
    raise RuntimeError(
        f"No sample with {label_col}={label_value!r} found in "
        f"{dataset_id}:{split}."
    )


def _resolve_plantdoc(name: str) -> Path:
    label, rel = PLANTDOC_TARGETS[name]
    local = _resolve_local(PLANTDOC_LOCAL_ROOT, rel)
    if local is not None:
        return local
    out = DEMO_SCRATCH / f"plantdoc__{name}.jpg"
    return _fetch_from_hf(
        "ankit-iiitdmj/iks-plantdoc", "test", "label", label, out,
    )


def _resolve_phantomfs(name: str) -> Path:
    label, rel = PHANTOMFS_TARGETS[name]
    local = _resolve_local(PHANTOMFS_LOCAL_ROOT, rel)
    if local is not None:
        return local
    out = DEMO_SCRATCH / f"phantomfs__{name}.jpg"
    return _fetch_from_hf(
        "ankit-iiitdmj/iks-soil-phantomfs", "train", "class_name", label, out,
    )


print("=== Resolving demo image sources (local repo → HF Hub fallback) ===")
DEMO_SAMPLES = [
    {
        "name": "peach_alluvial_soil_driven",
        "leaf_path": _resolve_plantdoc("peach_leaf"),
        "soil_path": _resolve_phantomfs("alluvial_soil"),
        "crop": "peach",
        "pathway": CausalPathway.SOIL_DRIVEN,
    },
    {
        "name": "grape_black_pest_vector",
        "leaf_path": _resolve_plantdoc("grape_leaf"),
        "soil_path": _resolve_phantomfs("black_soil"),
        "crop": "grape",
        "pathway": CausalPathway.PEST_VECTOR,
    },
    {
        "name": "corn_red_unknown",
        "leaf_path": _resolve_plantdoc("corn_rust_leaf"),
        "soil_path": _resolve_phantomfs("red_soil"),
        "crop": "corn",
        "pathway": CausalPathway.UNKNOWN,
    },
]

print()
print("=== Demo sample sources + predicted disease names ===")
sample_disease_names = []
for s in DEMO_SAMPLES:
    for kind in ("leaf_path", "soil_path"):
        p = Path(s[kind])
        assert p.is_file(), (
            f"Demo image missing: {p}  "
            f"(Phase 9 refuses to render Grad-CAM over a stand-in)"
        )
        assert p.stat().st_size > 0, f"Demo image is empty: {p}"
    # Predict per-sample so the supervisor can see distinct labels.
    pred = disease_engine.predict(Image.open(s["leaf_path"])).prediction
    sample_disease_names.append(pred.class_name)
    print(f"- {s['name']}")
    print(f"    leaf src     : {s['leaf_path']}")
    print(f"    soil src     : {s['soil_path']}")
    print(f"    crop         : {s['crop']}")
    print(f"    pred disease : {pred.class_name}  (idx={pred.class_index}  conf={pred.confidence:.3f})")

assert len(set(sample_disease_names)) >= 2, (
    "All three samples predicted the same disease class "
    f"{sample_disease_names!r} — Grad-CAM figures would explain the same "
    "label three times. Pick distinct PlantDoc test images that the model "
    "actually attends to differently."
)
print(f"\nSample disease labels are distinct: {sample_disease_names}")


In [ ]:
# Cell 8 — Disease Grad-CAM for the 3 multimodal demo samples (crop-first, heatmap-only-for-disease).
# Uses the shared disease_explain(): YOLO-crop -> C-PD predict -> eigen Grad-CAM if diseased,
# or a plain crop (no heatmap) if healthy. Populates disease_cams for the saved panels (Cell 11).
import matplotlib.pyplot as plt
from PIL import Image

disease_cams: dict[str, "GradCAMResult"] = {}
disease_crops: dict[str, "Image.Image"] = {}

fig, axes = plt.subplots(len(DEMO_SAMPLES), 2, figsize=(9, 4.3 * len(DEMO_SAMPLES)))
if len(DEMO_SAMPLES) == 1:
    axes = axes.reshape(1, 2)

for row, sample in enumerate(DEMO_SAMPLES):
    crop, _found = cropper.crop(Image.open(sample["leaf_path"]).convert("RGB"))
    cam, healthy = disease_explain(crop)
    disease_cams[sample["name"]] = cam
    disease_crops[sample["name"]] = crop

    axes[row, 0].imshow(crop)
    axes[row, 0].set_title(f"{sample['name']}\n(YOLO leaf crop)", fontsize=10)
    axes[row, 0].axis("off")
    axes[row, 1].imshow(cam.overlay_rgb)
    if healthy:
        axes[row, 1].set_title(f"✓ HEALTHY — {cam.pred_label} ({cam.pred_conf:.0%})\nno heatmap",
                               color="green", fontsize=10)
    else:
        axes[row, 1].set_title(f"⚠ DISEASE — {cam.pred_label} ({cam.pred_conf:.0%})\nGrad-CAM = lesion",
                               color="darkred", fontsize=10)
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()
print(f"\nGenerated {len(disease_cams)} disease panels (heatmap only where a disease was predicted).")


In [ ]:
# Cell 9 — Soil Grad-CAM × 3 heads per sample.
# Each head is wrapped via SoilHeadWrapper so pytorch_grad_cam sees a
# single-tensor forward. soil_gradcam runs the model in eval mode,
# pulls the per-head argmax label from the engine (so the explanation
# targets the SAME label the rest of the pipeline saw), and returns
# the overlay + heatmap + label/conf.
import matplotlib.pyplot as plt
from PIL import Image

from src.explain.gradcam import SOIL_HEADS, soil_gradcam

soil_cams_per_sample: dict[str, dict] = {}

for sample in DEMO_SAMPLES:
    print(f"--- {sample['name']} ---")
    cams = {}
    for head in SOIL_HEADS:
        cam = soil_gradcam(sample["soil_path"], soil_engine, head=head)
        cams[head] = cam
        print(f"  {head:<10} idx={cam.pred_index}  label={cam.pred_label!r}  conf={cam.pred_conf:.2f}")
    soil_cams_per_sample[sample["name"]] = cams

    # Render the 4-tile row: original soil + 3 head heatmaps.
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    soil_img = Image.open(sample["soil_path"]).convert("RGB")
    soil_img = soil_img.resize((soil_engine.config.image_size, soil_engine.config.image_size))
    axes[0].imshow(soil_img)
    axes[0].set_title(f"{sample['name']} (original)")
    axes[0].axis("off")
    for col, head in enumerate(SOIL_HEADS, start=1):
        cam = cams[head]
        axes[col].imshow(cam.overlay_rgb)
        axes[col].set_title(f"{head}\n{cam.pred_label} ({cam.pred_conf:.2f})")
        axes[col].axis("off")
    plt.tight_layout()
    plt.show()

print(f"\nGenerated {sum(len(c) for c in soil_cams_per_sample.values())} soil-head Grad-CAM overlays.")


In [ ]:
# Cell 10 — Build the query (Phase 8 Strategy A) and explain top-k retrieval.
# Per sample: build the multimodal context, render Strategy A's
# template query, retrieve top-5, then expand each chunk with the
# matched-query-term overlay.
from PIL import Image

from src.explain.chunk_highlight import explain_chunks
from src.integration import (
    TemplateStrategy,
    build_multimodal_context,
)
from src.integration.config import TemplateStrategyConfig

template_strategy = TemplateStrategy(TemplateStrategyConfig())

retrieval_per_sample: dict[str, dict] = {}

for sample in DEMO_SAMPLES:
    ctx = build_multimodal_context(
        leaf_image=Image.open(sample["leaf_path"]),
        soil_image=Image.open(sample["soil_path"]),
        crop_type=sample["crop"],
        causal_pathway=sample["pathway"],
        disease_engine=disease_engine,
        soil_engine=soil_engine,
        capture_embeddings=False,    # Strategy C is NOT exercised here.
    )
    query = template_strategy.build_query(ctx)
    retrieved = rag_pipeline.retriever.retrieve(query, k=5)
    explained = explain_chunks(query, retrieved)
    retrieval_per_sample[sample["name"]] = {
        "query": query, "retrieved": retrieved, "explained": explained,
    }
    print("=" * 78)
    print(f"SAMPLE: {sample['name']}")
    print(f"  QUERY: {query!r}")
    for row in explained:
        print(
            f"  #{row.rank}  score={row.score:.3f}  "
            f"{row.source_text} ch.{row.chapter} v.{row.verse_or_section}"
        )
        print(f"      matched: {row.matched_terms or '(no overlap)'}")


In [ ]:
# Cell 11 — Combined figures saved to results/explainability/<sample>/
# These PNGs are what the Phase 10 Streamlit UI will surface and the
# paper figures will reuse. The directory is committed to the repo
# (results/ is for tracked PNGs / metrics per master plan §41).
from pathlib import Path

import numpy as np
from PIL import Image

from src.explain.visualize import (
    DEFAULT_OUT_ROOT,
    render_retrieval_panel,
    render_vision_panel,
    save_explanation,
)

saved_paths: list[tuple[Path, Path]] = []

for sample in DEMO_SAMPLES:
    name = sample["name"]
    disease_cam = disease_cams[name]
    soil_cams = soil_cams_per_sample[name]

    # Show the YOLO leaf CROP (what the C-PD model + Grad-CAM actually saw),
    # so the panel's leaf image matches the crop-based heatmap.
    leaf = disease_crops.get(name)
    if leaf is None:
        leaf = Image.open(sample["leaf_path"]).convert("RGB")
    leaf = leaf.resize((disease_engine.image_size, disease_engine.image_size))
    leaf_rgb = np.asarray(leaf, dtype=np.uint8)

    soil = Image.open(sample["soil_path"]).convert("RGB")
    soil = soil.resize((soil_engine.config.image_size, soil_engine.config.image_size))
    soil_rgb = np.asarray(soil, dtype=np.uint8)

    vision_fig = render_vision_panel(
        sample_name=name,
        original_leaf=leaf_rgb,
        disease_cam=disease_cam,
        original_soil=soil_rgb,
        soil_cams=soil_cams,
    )
    retrieval_fig = render_retrieval_panel(
        sample_name=name,
        query=retrieval_per_sample[name]["query"],
        explained_chunks=retrieval_per_sample[name]["explained"],
    )
    paths = save_explanation(
        sample_name=name,
        vision_fig=vision_fig,
        retrieval_fig=retrieval_fig,
    )
    saved_paths.append(paths)
    print(f"{name:<40s} → {paths[0].name}, {paths[1].name}")

print()
print(f"All figures saved under {DEFAULT_OUT_ROOT.relative_to(REPO_PATH)}")
print()

# Inline preview so the supervisor doesn't have to dig into the Colab
# file browser. ``save_explanation`` already closed the matplotlib
# figures, so we re-load the PNGs from disk via IPython.display.
from IPython.display import Image as IPImage
from IPython.display import display

for v_path, r_path in saved_paths:
    print(f"=== {v_path.parent.name} ===")
    print("  vision panel:")
    display(IPImage(filename=str(v_path)))
    print("  retrieval panel:")
    display(IPImage(filename=str(r_path)))


## What the figures should show + what's deferred

**What to look for in the saved panels**

- ``results/explainability/<sample>/vision_panel.png`` — the disease
  heatmap should concentrate over the lesion / discoloured region
  (NOT the corner or the background). The three soil-head heatmaps
  should attend to different regions: ``soil_type`` typically focuses
  on the colour-rich area; ``moisture`` on glossier / damper-looking
  patches; ``texture`` on the granular surface itself. If all three
  soil-head heatmaps look identical, the multi-task head is
  collapsing — flag for retraining (out of Phase 9 scope).
- ``results/explainability/<sample>/retrieval_panel.png`` — the
  similarity bar chart should be monotone descending (it's the
  retrieved top-k by score). The matched terms next to each chunk
  tell the supervisor *why* the retriever picked the chunk; chunks
  with ``(no overlap)`` are pure dense-retrieval hits — those are
  expected and not a bug, but worth flagging when they dominate.

**What this notebook deliberately does NOT do**

- **Pointing-game / faithfulness benchmarks.** Quantitative
  Grad-CAM evaluation (how well does the heatmap localise the GT
  lesion mask?) needs segmentation labels we don't have. Phase 11
  will revisit this with the gold-query set.
- **Answer-grounded chunk highlighting.** Phase 9 explains the
  retrieval step only; aligning chunks to the LLM's *answer*
  sentences mixes retrieval and generation failure modes. The
  generation-side audit comes via RAGAS faithfulness in Phase 11.
- **Sentence-level chunk highlighting inside the answer.** Possible
  to add later, but the demo gets more mileage out of the
  retrieval panel + Grad-CAM than a third explanation surface.

**What comes next**

- **Phase 10** — Streamlit UI that surfaces the Grad-CAM panels and
  the retrieval panel live. The figures rendered here are the
  components that UI will display; the saved PNGs are the paper's
  reference figures.
- **Phase 11** — rigorous evaluation: RAGAS context_precision /
  context_recall on the gold-query set, faithfulness on the
  generated answers, and a quantitative Grad-CAM pointing-game if a
  segmentation mask subset of PlantDoc becomes available.
